# 1349. Maximum Students Taking Exam

## Topic Alignment
- This problem models resource allocation with constraints, appearing in task scheduling, server placement, and optimal layout design in data centers.
- Bitmask DP with row-by-row processing is essential for grid-based optimization in computer vision (image segmentation), VLSI design, and warehouse optimization.
- The technique of validating states with bitwise operations is crucial for constraint satisfaction problems in production systems.

## Metadata Summary
- Source: https://leetcode.com/problems/maximum-students-taking-exam/
- Tags: Dynamic Programming, Bit Manipulation, Matrix, State Compression
- Difficulty: Hard
- Priority: High

## Problem Statement
Given a `m × n` matrix `seats` that represent seats in a classroom. If a seat is broken, it is denoted by `'#'` otherwise it is denoted by `'.'`.

Students can see the answers of those sitting next to the left, right, upper left and upper right, but cannot see the answers of the student sitting directly in front or behind.

Return the **maximum** number of students that can take the exam together without any cheating being possible.

**Constraints**:
- `seats` contains only characters `'.'` and `'#'`.
- `m == seats.length`
- `n == seats[i].length`
- `1 <= m <= 8`
- `1 <= n <= 8`

## Progressive Hints
- Hint 1: Process the classroom row by row from top to bottom.
- Hint 2: Use bitmask to represent which seats in a row are occupied (m, n <= 8 → feasible).
- Hint 3: For each row, generate all valid seating arrangements (no adjacent students, no broken seats).
- Hint 4: State: `dp[row][mask]` = max students in rows 0..row with row 'row' having arrangement 'mask'.
- Hint 5: Validate mask: no adjacent bits set (no left/right neighbors).
- Hint 6: Validate transition: current row mask compatible with previous row (no upper-left/upper-right).
- Hint 7: For mask with no adjacent bits: `(mask & (mask << 1)) == 0` and `(mask & (mask >> 1)) == 0`.
- Hint 8: Count students in mask: `bin(mask).count('1')`.

## Solution Overview
Use **row-by-row bitmask DP**.

**State Definition**:
- `dp[i][mask]`: maximum students when processing rows 0..i and row i has seating arrangement `mask`
- `mask`: bitmask where bit j = 1 means seat j is occupied

**Valid Mask Constraints**:
1. No broken seats: `mask & broken_mask == 0`
2. No adjacent students: `mask & (mask << 1) == 0`

**Valid Transition Constraints**:
Given current row mask `cur` and previous row mask `prev`:
1. No upper-left cheating: `cur & (prev << 1) == 0`
2. No upper-right cheating: `cur & (prev >> 1) == 0`

**Recurrence**:
```python
For each row i:
    For each valid mask cur for row i:
        For each valid mask prev for row i-1:
            if compatible(cur, prev):
                dp[i][cur] = max(dp[i][cur], 
                                 dp[i-1][prev] + count_bits(cur))
```

**Answer**: `max(dp[m-1][mask])` for all masks

## Detailed Explanation

### Problem Constraints Visualization

**Cheating directions**:
```
    UL  U  UR
     \  |  /
  L - Student - R
      /  |  \\
    DL  D  DR
```

Students can cheat from: **L, R, UL, UR** (4 directions)

**We must prevent**:
- Left-right: no adjacent students in same row
- Upper-left: no student at (i-1, j-1) when student at (i, j)
- Upper-right: no student at (i-1, j+1) when student at (i, j)

Front and back are OK (can't see those directions).

---

### Bitmask Representation

**Example row**: `. . # . .` (seats 0,1,3,4 available, seat 2 broken)

**Possible valid masks**:
- `0b00000` (0): no students
- `0b00001` (1): student at seat 0 only
- `0b01000` (8): student at seat 3 only
- `0b10000` (16): student at seat 4 only
- `0b01001` (9): students at seats 0 and 3
- `0b10001` (17): students at seats 0 and 4
- etc.

**Invalid masks**:
- `0b00011` (3): adjacent students at seats 0 and 1 ✗
- `0b00100` (4): student on broken seat ✗
- `0b01100` (12): student on broken seat + adjacent ✗

---

### Validating a Mask

**Check 1: No adjacent students**
```python
def has_adjacent(mask):
    return (mask & (mask << 1)) != 0
```

Example:
- mask = `0b01010` (students at positions 1, 3)
- mask << 1 = `0b10100`
- mask & (mask << 1) = `0b00000` → no adjacent ✓

- mask = `0b00110` (students at positions 1, 2)
- mask << 1 = `0b01100`
- mask & (mask << 1) = `0b00100` ≠ 0 → has adjacent ✗

**Check 2: No broken seats used**
```python
def uses_broken_seats(mask, broken_mask):
    return (mask & broken_mask) != 0
```

---

### Validating Transitions

**Check upper-left conflict**:
```python
def upper_left_conflict(cur_mask, prev_mask):
    # If prev_mask has student at j, prev_mask bit j is set
    # If cur_mask has student at j+1, we have conflict
    # prev_mask << 1 shifts prev students right
    # If overlap with cur_mask, there's upper-left conflict
    return (cur_mask & (prev_mask << 1)) != 0
```

**Check upper-right conflict**:
```python
def upper_right_conflict(cur_mask, prev_mask):
    # If prev_mask has student at j, prev_mask bit j is set
    # If cur_mask has student at j-1, we have conflict
    # prev_mask >> 1 shifts prev students left
    return (cur_mask & (prev_mask >> 1)) != 0
```

**Combined check**:
```python
def compatible(cur, prev):
    return ((cur & (prev << 1)) == 0 and 
            (cur & (prev >> 1)) == 0)
```

---

### DP Algorithm

**Preprocessing**: Generate all valid masks for each row
```python
valid_masks[i] = [masks that are valid for row i]
# Valid: no adjacent, no broken seats
```

**DP iteration**:
```python
# Base case: row 0
for mask in valid_masks[0]:
    dp[0][mask] = count_bits(mask)

# Fill subsequent rows
for i in range(1, m):
    for cur_mask in valid_masks[i]:
        for prev_mask in valid_masks[i-1]:
            if compatible(cur_mask, prev_mask):
                dp[i][cur_mask] = max(
                    dp[i][cur_mask],
                    dp[i-1][prev_mask] + count_bits(cur_mask)
                )
```

**Answer**:
```python
return max(dp[m-1].values())
```

---

### Example Walkthrough

**Input**:
```
[["#",".","#","#",".","#"],
 [".","#","#","#","#","."],
 ["#",".","#","#",".","#"]]
```

**Row 0**: `# . # # . #`
- Available seats: positions 1, 4
- Valid masks: 0 (none), 0b000010 (pos 1), 0b010000 (pos 4), 0b010010 (both)
- `dp[0][0b010010] = 2`

**Row 1**: `. # # # # .`
- Available seats: positions 0, 5
- Valid masks: 0, 0b000001, 0b100000, 0b100001
- Check compatibility with row 0:
  - cur=0b100001 (pos 0, 5), prev=0b010010 (pos 1, 4)
  - Upper-left: cur & (prev << 1) = 0b100001 & 0b100100 = 0b100000 ≠ 0 ✗
  - Conflict at position 5!

After careful checking, optimal might be 4 students.

---

### Optimization: Precompute Valid Masks

```python
def get_valid_masks(row, n):
    valid = []
    # Try all 2^n possible masks
    for mask in range(1 << n):
        # Check no broken seats
        ok = True
        for j in range(n):
            if (mask >> j) & 1:  # Seat j occupied
                if row[j] == '#':  # Broken
                    ok = False
                    break
        
        if not ok:
            continue
        
        # Check no adjacent
        if mask & (mask << 1):
            continue
        
        valid.append(mask)
    return valid
```

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Bitmask DP | O(m × 4^n) | O(m × 2^n) | Optimal for small m, n |
| Backtracking | O(2^(m×n)) | O(m×n) | Exponential, impractical |
| Greedy | O(m×n) | O(1) | Incorrect, doesn't handle constraints |
| ILP solver | Varies | Varies | Correct but overkill for small inputs |

In [ ]:
from typing import List

class Solution:
    def maxStudents(self, seats: List[List[str]]) -> int:
        """
        Bitmask DP for maximum students without cheating.
        
        Time: O(m × 4^n) - m rows, each with 2^n × 2^n transitions
        Space: O(m × 2^n) - DP table
        """
        m, n = len(seats), len(seats[0])
        
        def is_valid_mask(mask, row):
            """Check if mask is valid for given row."""
            # No adjacent students
            if mask & (mask << 1):
                return False
            
            # No broken seats used
            for j in range(n):
                if (mask >> j) & 1:  # Seat j occupied
                    if seats[row][j] == '#':  # Broken seat
                        return False
            
            return True
        
        def compatible(cur_mask, prev_mask):
            """Check if cur_mask and prev_mask are compatible."""
            # No upper-left cheating
            if cur_mask & (prev_mask << 1):
                return False
            
            # No upper-right cheating
            if cur_mask & (prev_mask >> 1):
                return False
            
            return True
        
        def count_students(mask):
            """Count number of students in mask."""
            return bin(mask).count('1')
        
        # Precompute valid masks for each row
        valid_masks = []
        for i in range(m):
            masks = []
            for mask in range(1 << n):
                if is_valid_mask(mask, i):
                    masks.append(mask)
            valid_masks.append(masks)
        
        # DP: dp[i][mask] = max students in rows 0..i with row i = mask
        dp = [{} for _ in range(m)]
        
        # Base case: row 0
        for mask in valid_masks[0]:
            dp[0][mask] = count_students(mask)
        
        # Fill subsequent rows
        for i in range(1, m):
            for cur_mask in valid_masks[i]:
                dp[i][cur_mask] = 0
                for prev_mask in valid_masks[i-1]:
                    if compatible(cur_mask, prev_mask):
                        dp[i][cur_mask] = max(
                            dp[i][cur_mask],
                            dp[i-1][prev_mask] + count_students(cur_mask)
                        )
        
        # Return maximum across all masks in last row
        return max(dp[m-1].values()) if dp[m-1] else 0

In [ ]:
# Test cases
tests = [
    ([["#",".","#","#",".","#"],
      [".","#","#","#","#","."],
      ["#",".","#","#",".","#"]], 4),
    ([[".","#"],
      ["#","#"],
      ["#","."]], 1),
    ([["#",".",".",".","#"],
      [".","#",".","#","."],
      [".",".","#",".","."],
      [".","#",".","#","."],
      ["#",".",".",".","#"]], 10),
]

solver = Solution()
for seats, expected in tests:
    result = solver.maxStudents(seats)
    assert result == expected, f"Failed: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(m × 2^n × 2^n) = O(m × 4^n)
  - m rows
  - For each row, try O(2^n) current masks
  - For each current mask, check O(2^n) previous masks
  - For m=8, n=8: 8 × 4^8 = 524,288 operations (fast)
- **Space**: O(m × 2^n)
  - DP table stores at most 2^n masks per row
  - For m=8, n=8: 8 × 256 = 2,048 entries

## Edge Cases & Pitfalls
- **All broken seats**: Return 0
- **Single row**: Check adjacent constraint only
- **Single column**: Each row independent, sum of valid students
- **Checkerboard pattern**: Maximum density possible
- **No broken seats**: More valid configurations
- **Common mistake**: Forgetting to check upper-right (only checking upper-left)
- **Common mistake**: Not handling shift operations correctly (bits shifting out of range)
- **Common mistake**: Forgetting that students can't see front/back (only 4 directions matter)
- **Bit manipulation**: Be careful with shift operations at boundaries (<<, >> may overflow)

## Follow-up Variants
- **Different cheating directions**: Allow/disallow different visibility patterns
- **Minimize proctors**: Place minimum proctors to monitor all students
- **Weighted students**: Students have different values, maximize total value
- **3D classroom**: Multiple floors, add vertical constraints
- **Dynamic seats**: Seats can be repaired, find optimal repair strategy
- **Probabilistic**: Each student has probability of cheating, minimize expected cheating
- **Online version**: Students arrive one by one, assign seats dynamically
- **Rectangular constraints**: Groups must form rectangles

## Takeaways
- **Row-by-row bitmask DP** is powerful for grid problems with local constraints.
- **Bitwise operations** elegantly encode constraint checking (adjacent, diagonal).
- Separating **validity** (within row) from **compatibility** (between rows) simplifies logic.
- **Precomputing valid masks** reduces redundant validation checks.
- Understanding **shift operations** for diagonal checking is key (`mask << 1`, `mask >> 1`).
- This pattern extends to tiling problems, graph coloring on grids, and resource allocation.
- The technique is practical for small grids (m, n <= 20) but becomes infeasible for larger inputs.
- **State compression** trades memory efficiency for compact representation of configurations.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 1240 | Tiling a Rectangle with the Fewest Squares | Bitmask DP on grid |
| LC 1659 | Maximize Grid Happiness | Bitmask DP with state |
| LC 1723 | Find Minimum Time to Finish All Jobs | State compression |
| LC 847 | Shortest Path Visiting All Nodes | Bitmask DP on graph |
| LC 1595 | Minimum Cost to Connect Two Groups | Bitmask matching |
| LC 996 | Number of Squareful Arrays | Bitmask permutation DP |